# Тема 1. Вариант 4
Борисюк Е. Г.
$$z=3x_1+x_2+4x_3+2x_4\to\min$$
$$2x_1+x_2+x_3\le8,\quad x_1+x_2+x_4\ge6,\quad x_i\ge0.$$

### Канонический вид и вспомогательная задача
Добавляю переменные запаса и избытка. Начальный базис дополняю искусственными переменными только там, где нет подходящего столбца запаса. Для варианта 4 получается цель $w=a_2$.

In [1]:
from fractions import Fraction as F

def canonicalize(A, b, senses, c, direction='min'):
    A = [[F(v) for v in row] for row in A]
    b = list(map(F, b)); senses = list(senses)
    n = len(c); names = [f'x{i+1}' for i in range(n)]
    for i in range(len(b)):
        if b[i] < 0:
            A[i] = [-v for v in A[i]]; b[i] *= -1
            senses[i] = {'<=':'>=','>=':'<=','=':'='}[senses[i]]
    for i,s in enumerate(senses):
        if s not in ('<=','>=','='): raise ValueError('Unknown relation')
        if s != '=':
            for j,row in enumerate(A): row.append(F((1 if s=='<=' else -1) if i==j else 0))
            names.append(f's{i+1}')
    cost = [F(v)*(1 if direction=='min' else -1) for v in c] + [F(0)]*(len(names)-n)
    return A,b,cost,names

def auxiliary_problem(A,b,names):
    m=len(b); n=len(names); basis=[]; missing=[]
    for i in range(m):
        cols=[j for j,name in enumerate(names) if name.startswith('s') and
              all(A[k][j]==int(k==i) for k in range(m))]
        if cols: basis.append(cols[0])
        else: basis.append(n+len(missing)); missing.append(i)
    T=[row+[F(int(i==k)) for k in missing]+[b[i]] for i,row in enumerate(A)]
    return T,basis,[F(0)]*n+[F(1)]*len(missing),names+[f'a{k+1}' for k in missing]



### Симплекс-шаг
Считаю оценки $r_j=c_j-c_B^T T_j$. При отрицательной оценке выбираю входящий столбец, затем строку по минимальному отношению $b_i/t_{ik}$ при $t_{ik}>0$. Пересчитываю таблицу и сохраняю шаг.

In [2]:
def reduced_costs(T,basis,cost):
    r=[cost[j]-sum(cost[basis[i]]*T[i][j] for i in range(len(T))) for j in range(len(cost))]
    z=sum(cost[basis[i]]*T[i][-1] for i in range(len(T)))
    return r,z

def pivot(T,basis,row,col):
    p=T[row][col]; T[row]=[v/p for v in T[row]]
    for i in range(len(T)):
        if i!=row:
            k=T[i][col]; T[i]=[v-k*w for v,w in zip(T[i],T[row])]
    basis[row]=col

def simplex(T,basis,cost,names,phase,history):
    for step in range(1000):
        r,z=reduced_costs(T,basis,cost)
        entry={'phase':phase,'step':step,'names':names[:],'basis':[names[i] for i in basis],
               'T':[[str(v) for v in row] for row in T],'r':[str(v) for v in r],'z':str(z)}
        history.append(entry)
        cols=[j for j,v in enumerate(r) if v<0]
        if not cols: return
        col=min(cols)
        rows=[i for i in range(len(T)) if T[i][col]>0]
        if not rows: raise ValueError('Objective unbounded')
        row=min(rows,key=lambda i:(T[i][-1]/T[i][col],basis[i]))
        entry['enter']=names[col]; entry['leave']=names[basis[row]]
        entry['ratios']=[str(T[i][-1]/T[i][col]) if T[i][col]>0 else '—' for i in range(len(T))]
        pivot(T,basis,row,col)
    raise RuntimeError('Iteration limit')



### Переход к основной задаче
Если минимум вспомогательной цели равен нулю, удаляю искусственные переменные и возвращаю исходную цель. Когда все оценки неотрицательны, считываю оптимальный план из базиса.

In [3]:
def main_problem(T,basis,n):

    for i in range(len(T)-1,-1,-1):
        if basis[i]>=n:
            if T[i][-1]!=0: raise ValueError('Infeasible')
            cols=[j for j in range(n) if j not in basis and T[i][j]!=0]
            if cols: pivot(T,basis,i,min(cols))
            else: T.pop(i); basis.pop(i)
    return [row[:n]+[row[-1]] for row in T],basis

def solve_lp(A,b,senses,c,direction='min'):
    C,rhs,cost,names=canonicalize(A,b,senses,c,direction)
    T,basis,aux_cost,aux_names=auxiliary_problem(C,rhs,names)
    history=[]; simplex(T,basis,aux_cost,aux_names,'I',history)
    if reduced_costs(T,basis,aux_cost)[1]!=0: raise ValueError('Infeasible')
    T,basis=main_problem(T,basis,len(names))
    simplex(T,basis,cost,names,'II',history)
    x=[F(0)]*len(names)
    for i,j in enumerate(basis): x[j]=T[i][-1]
    return x[:len(c)],sum(F(v)*u for v,u in zip(c,x)),history



### Вариант 4
Задаю коэффициенты исходной задачи. Обе фазы, промежуточные таблицы и ответ выводятся ниже.

In [4]:
A = [[2, 1, 1, 0], [1, 1, 0, 1]]
b = [8, 6]
senses = ['<=', '>=']
c = [3, 1, 4, 2]
x, z, history = solve_lp(A, b, senses, c)

for t in history:
    print('Фаза', t['phase'], 'шаг', t['step'])
    print('Базис | ' + ' | '.join(t['names']) + ' | b')
    for name, row in zip(t['basis'], t['T']):
        print(name + ' | ' + ' | '.join(row))
    print('r | ' + ' | '.join(t['r']), 'Цель:', t['z'])
    if 'enter' in t:
        print('Входит:', t['enter'], 'Выходит:', t['leave'])
    print()
print('x* =', tuple(map(str, x)))
print('z =', z)


Фаза I шаг 0
Базис | x1 | x2 | x3 | x4 | s1 | s2 | a2 | b
s1 | 2 | 1 | 1 | 0 | 1 | 0 | 0 | 8
a2 | 1 | 1 | 0 | 1 | 0 | -1 | 1 | 6
r | -1 | -1 | 0 | -1 | 0 | 1 | 0 Цель: 6
Входит: x1 Выходит: s1

Фаза I шаг 1
Базис | x1 | x2 | x3 | x4 | s1 | s2 | a2 | b
x1 | 1 | 1/2 | 1/2 | 0 | 1/2 | 0 | 0 | 4
a2 | 0 | 1/2 | -1/2 | 1 | -1/2 | -1 | 1 | 2
r | 0 | -1/2 | 1/2 | -1 | 1/2 | 1 | 0 Цель: 2
Входит: x2 Выходит: a2

Фаза I шаг 2
Базис | x1 | x2 | x3 | x4 | s1 | s2 | a2 | b
x1 | 1 | 0 | 1 | -1 | 1 | 1 | -1 | 2
x2 | 0 | 1 | -1 | 2 | -1 | -2 | 2 | 4
r | 0 | 0 | 0 | 0 | 0 | 0 | 1 Цель: 0

Фаза II шаг 0
Базис | x1 | x2 | x3 | x4 | s1 | s2 | b
x1 | 1 | 0 | 1 | -1 | 1 | 1 | 2
x2 | 0 | 1 | -1 | 2 | -1 | -2 | 4
r | 0 | 0 | 2 | 3 | -2 | -1 Цель: 10
Входит: s1 Выходит: x1

Фаза II шаг 1
Базис | x1 | x2 | x3 | x4 | s1 | s2 | b
s1 | 1 | 0 | 1 | -1 | 1 | 1 | 2
x2 | 1 | 1 | 0 | 1 | 0 | -1 | 6
r | 2 | 0 | 4 | 1 | 0 | 1 Цель: 6

x* = ('0', '6', '0', '0')
z = 6
